In [1]:

!pip install findspark

Defaulting to user installation because normal site-packages is not writeable


In [1]:
import findspark
findspark.init()

In [2]:
import warnings
warnings.filterwarnings('ignore')
spark_ui_port = 4040
app_name = "Otus"

In [3]:
import pyspark
from pyspark import SparkContext
from pyspark.sql.types import StringType, DoubleType, IntegerType, StructType, StructField
import pyspark.sql.functions as F
from pyspark.sql.functions import isnan, when, count, col

spark = (
    pyspark.sql.SparkSession
        .builder
        .appName(app_name)
        .config("spark.executor.memory", "8g")
        .config("spark.driver.memory", "4g")
        .config("spark.executor.cores", 4)
        .config("spark.executor.instances", 3)
        .getOrCreate()
)
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)  # to pretty print pyspark.DataFrame in jupyter
spark.conf.set("hadoop.fs.defaultFS", "hdfs:/rc1d-dataproc-m-jk2lthu6rsbgtvkd.mdb.yandexcloud.net:8888/")

In [4]:
schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("tx_datetime", StringType()),
    StructField("customer_id", IntegerType()),
    StructField("terminal_id", IntegerType()),
    StructField("tx_amount", DoubleType()),
    StructField("tx_time_seconds", IntegerType()),
    StructField("tx_time_days", IntegerType()),
    StructField("tx_fraud", IntegerType()),
    StructField("tx_fraud_scenario", IntegerType())
])

In [5]:
df = spark.read.csv('/user/ubuntu/data/2022-11-04.txt', schema=schema, header=True)

In [6]:
df.show(5)

+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|transaction_id|        tx_datetime|customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|    1832792610|2022-11-04 14:22:18|          0|         53|    63.58|      101139738|        1170|       0|                0|
|    1832792611|2022-11-04 02:12:24|          0|         53|    92.95|      101095944|        1170|       0|                0|
|    1832792612|2022-11-04 12:49:35|          3|        205|    48.88|      101134175|        1170|       0|                0|
|    1832792613|2022-11-04 02:40:01|          5|        383|    24.69|      101097601|        1170|       0|                0|
|    1832792614|2022-11-04 08:02:05|          6|        858|    95.48|      101116925|        1170|       0|   

In [7]:
spark.read.csv('/user/ubuntu/data/2019-08-22.txt', schema=schema, header=True)\
.repartition(1).write.mode('overwrite').parquet('/user/ubuntu/data/dq_data')

In [5]:
df = spark.read.parquet('/user/ubuntu/data/dq_data')

In [6]:
df.show(5)

+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|transaction_id|        tx_datetime|customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|             0|2019-08-22 06:51:03|          0|        711|    70.91|          24663|           0|       0|                0|
|             1|2019-08-22 05:10:37|          0|          0|    90.55|          18637|           0|       0|                0|
|             2|2019-08-22 19:05:33|          0|        753|    35.38|          68733|           0|       0|                0|
|             3|2019-08-22 07:21:33|          0|          0|    80.41|          26493|           0|       0|                0|
|             4|2019-08-22 09:06:17|          1|        981|   102.83|          32777|           0|       0|   

In [7]:
df.count()

46998983

In [ ]:
# 1. Проверка на наличие пустых значений
null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show()
# В одном из файлов нашел пропуски в поле termanal_id. Буду заполнять средним

+--------------+-----------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|transaction_id|tx_datetime|customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+--------------+-----------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|             0|          0|          0|       2298|        0|              0|           0|       0|                0|
+--------------+-----------+-----------+-----------+---------+---------------+------------+--------+-----------------+



In [9]:
# 2. Проверка на дубликаты
duplicate_count = df.count() - df.distinct().count()
print(f"Количество дубликатов: {duplicate_count}")

Количество дубликатов: 8


In [10]:
# 3. Проверка на уникальность значений в определенных столбцах
# Пример: проверка уникальности значений в столбце 'transaction_id'
unique_id_count = df.select("transaction_id").distinct().count()
total_id_count = df.count()
print(f"Количество уникальных 'transaction_id': {unique_id_count}, Общее количество 'transaction_id': {total_id_count}, разница: {total_id_count - unique_id_count}")

Количество уникальных 'transaction_id': 46998975, Общее количество 'transaction_id': 46998983, разница: 8


In [ ]:
# 4. Анализ мин, макс, среднее
df.describe()
# Вижу -999999 в customer_id, нужно убрать. Про остальные дннаые ничего не могу сказать. (Например значение 999999 в customer_id может быть технической записью. Нет ТЗ, поэтому оставляю. )

summary,transaction_id,tx_datetime,customer_id,terminal_id,tx_amount,tx_time_seconds,tx_time_days,tx_fraud,tx_fraud_scenario
count,46998983,46998983,46998983,46996685,46998983,46998983,46998983,46998983,46998983
mean,1.856292096624556E9,null,500437.420257349,28085.477290004605,54.17185979556221,1.0238400246815348E8,1184.5000234154004,0.02991875377388485,0.060363710423265965
stddev,1.3567435381965624E7,null,288558.3518378499,1571112.249845114,40.93842760395921,748058.0447052654,8.655529240607228,0.17036320777691383,0.34608998632486965
min,1832792610,2022-11-04 00:00:00,-999999,0,0.0,101088000,1170,0,0
max,1879791584,2022-12-03 24:00:00,999999,89518096,4754.5,103680000,1199,1,3


In [ ]:
# 5 Смотрю уникальные значения
df.select('tx_fraud_scenario').distinct()

tx_fraud_scenario
1
3
2
0


In [ ]:
# 5 Смотрю уникальные значения
df.select('tx_fraud').distinct()

tx_fraud
1
0


In [ ]:
# 6. Считаю среднее для terminal_id для заполнения
mean_value = int(df.select("terminal_id").agg({"terminal_id": "avg"}).first()[0])
mean_value

28085

In [ ]:
# 7. Финальная обработка
df_filtered = spark.read.csv('/user/ubuntu/data/', schema=schema, header=True).filter(col("customer_id") > 0).dropDuplicates().fillna({"terminal_id": mean_value})

In [38]:
spark.stop()

In [ ]:
# from subprocess import Popen, PIPE
# hdfs_path = '/user/ubuntu/data'
# process = Popen(f'hdfs dfs -ls -h {hdfs_path}', shell=True, stdout=PIPE, stderr=PIPE)
# std_out, std_err = process.communicate()
# list_of_file_names = [fn.split(' ')[-1].split('/')[-1] for fn in std_out.decode().split('\n')[1:]][:-1]
# list_of_file_names_with_full_address = [fn.split(' ')[-1] for fn in std_out.decode().split('\n')[1:]][:-1]

In [ ]:
# spark.read.csv(list_of_file_names_with_full_address[0], schema=schema, header=True)\
# .withColumn('file_name', F.lit(f'{list_of_file_names_with_full_address[0][-14:-4]}'))\
# .repartition(1).write\
# .partitionBy('file_name')\
# .mode('overwrite').parquet('/user/ubuntu/data/dq_data')
# for file in list_of_file_names_with_full_address[1:-1]:
#     spark.read.csv(f'{file}', schema=schema, header=True)\
#     .withColumn('file_name', F.lit(f'{file[-14:-4]}'))\
#     .repartition(1).write\
#     .partitionBy('file_name')\
#     .mode('append').parquet('/user/ubuntu/data/dq_data')
#     call(f'hdfs dfs -rm -r -skipTrash {file}', shell=True)